# NB04 — Input repair, then differential expression (GBM vs control)

**Why this version exists.** The earlier NB04 stopped with `fewer than 3 samples in a group`. The cause was upstream:
NB03 overwrote `GSE4290_samples.csv` and `GSE50161_samples.csv` with wrong group labels (all included samples were
labelled `GBM`, so limma saw zero controls). TCGA also had **no** usable survival patients, because age and IDH status
were missing (the GDC phenotype and mutation files returned HTTP 403).

**Part A — repair inputs (runs once, safe to re-run)**
* A1. Rebuild the GEO sample sheets from the raw series matrices with the same rules as NB02 and check the counts
  (expected: GSE4290 77 GBM / 23 epilepsy controls; GSE50161 34 GBM / 13 normal).
* A2. Add TCGA age and IDH status. Sources tried in order: (1) a manual file you place yourself, (2) the GDC API,
  (3) the cBioPortal API. A patient is called IDH-wildtype only if it was sequenced and has no IDH1 R132 / IDH2 R172 /
  IDH2 R140 variant. Not sequenced = unknown = excluded from the primary survival cohort.

**Part B — differential expression**
* Microarrays (GSE4290, GSE50161): `limma`, intensity-trend robust empirical Bayes.
* TCGA-GBM vs GTEx cortex (Toil counts): `edgeR` TMM + `limma-voom`, sex as covariate.
* Gene IDs are checked and converted to HGNC symbols before testing (probe IDs or Ensembl IDs are mapped).

Each cohort gives a full table (log2FC, SE, t, p, BH p) in `05_Differential_Expression/`, plus volcano plot, heatmap,
group summaries and a summary row.

**Caveat (H2):** each control type is a different comparison (epilepsy-surgery cortex, GSE50161 normal brain,
post-mortem GTEx cortex). They are never pooled. In the Toil comparison, study (TCGA vs GTEx) is confounded with group.

In [ ]:
import os, sys
ROOT = os.environ.get("GBM_ROOT", "/content/drive/MyDrive/GBM_MultiCohort_Research")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab; GBM_ROOT =", ROOT)
sys.path.insert(0, os.path.join(ROOT, "00_Project_Config"))
from gbm_common import *
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
RAW = P("raw"); MAT = P("meta", "matrices"); DE = CFG["de"]; MIN_N = CFG["preprocessing"]["min_group_n"]
DE_IN = P("de", "inputs")          # cleaned, symbol-level matrices written here for R
print("SMOKE =", SMOKE, "| FDR =", DE["fdr"], "| |log2FC| >=", DE["min_abs_log2fc"])

In [ ]:
# R packages needed here (installed by NB00; this only fills gaps in a fresh runtime).
import shutil
require(shutil.which("Rscript") is not None, "Rscript is missing. Run NB00 first.")
R_CHECK = r'''
options(repos = c(CRAN = "https://cloud.r-project.org"))
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
for (p in c("limma", "edgeR")) if (!requireNamespace(p, quietly = TRUE)) BiocManager::install(p, ask = FALSE, update = FALSE)
for (p in c("data.table", "statmod", "R.utils")) if (!requireNamespace(p, quietly = TRUE)) install.packages(p)
for (p in c("limma", "edgeR", "data.table", "statmod", "R.utils")) {
  if (!requireNamespace(p, quietly = TRUE)) stop("Missing R package: ", p)
  cat(p, as.character(packageVersion(p)), "\n")
}
'''
print(run_r(write_r("nb04_check_packages.R", R_CHECK), timeout=3600))

## Shared helper for gene IDs
Writes `gbm_extra.py` next to `gbm_common.py`. Later notebooks import `load_expr` from it so every matrix is read with
HGNC symbols as row names, whatever ID type NB03 wrote.

In [ ]:
EXTRA_SRC = r'''"""Gene-ID helpers (written by NB04). Every matrix is returned with HGNC symbols as row names."""
import numpy as np
import pandas as pd
from gbm_common import P, find_col, require_files


def probe_map_for(acc):
    f = P("meta", f"{acc}_probe_map.csv", mkdir=False)
    require_files([f], f"Probe IDs found for {acc} but no probe map; re-run the NB03 mapping step.")
    pm = pd.read_csv(f).astype(str)
    pcol = max(pm.columns, key=lambda c: pm[c].str.contains(r"_at$").mean())
    scol = max((c for c in pm.columns if c != pcol), key=lambda c: pm[c].str.match(r"^[A-Z][A-Z0-9\-\.]+$").mean())
    return dict(zip(pm[pcol], pm[scol]))


def to_symbols(expr, name, kind="log", quiet=False):
    """Return a symbol-indexed matrix. kind='log' keeps the highest-mean row per symbol; kind='counts' sums."""
    idx = pd.Index(expr.index.astype(str))
    if idx.str.match(r"^ENSG\d+").mean() > 0.5:
        pm = pd.read_csv(P("raw") / "TOIL" / "gencode.v23.annotation.gene.probemap", sep="\t")
        ids, syms = pm[find_col(pm, ["id"])].astype(str), pm[find_col(pm, ["gene"])].astype(str)
        mp = {**dict(zip(ids.str.split(".").str[0], syms)), **dict(zip(ids, syms))}
        new = [mp.get(i, mp.get(i.split(".")[0])) for i in idx]
        how = "Ensembl -> symbol (GENCODE v23)"
    elif idx.str.contains(r"_at$").mean() > 0.5:
        mp = probe_map_for(name)
        new = [mp.get(i) for i in idx]
        how = "probe -> symbol (NB03 probe map)"
    else:
        new, how = list(idx), "already symbols"
    e = expr.copy(); e.index = new
    e = e[e.index.notna() & ~pd.Index(e.index.astype(str)).isin(["", "nan", "None"])]
    if e.index.duplicated().any():
        if kind == "counts":
            e = e.groupby(level=0).sum()
        else:
            e = e.iloc[np.argsort(-e.mean(axis=1).values, kind="stable")]
            e = e[~e.index.duplicated(keep="first")]
    if not quiet:
        print(f"{name}: {how}; {expr.shape[0]} rows -> {e.shape[0]} genes")
    return e


def load_expr(path, kind="log", quiet=True):
    """Read a genes x samples CSV and return it symbol-indexed (Ensembl / probe IDs are mapped)."""
    e = pd.read_csv(path, index_col=0)
    return to_symbols(e, path.name.split("_")[0], kind, quiet=quiet)
'''
(P("config") / "gbm_extra.py").write_text(EXTRA_SRC)
if "gbm_extra" in sys.modules:
    del sys.modules["gbm_extra"]
from gbm_extra import to_symbols
print("gbm_extra.py written:", P("config") / "gbm_extra.py")

## A1. Rebuild the GEO sample sheets from the raw series matrices

In [ ]:
import re

GBM_RE = r"\bglioblastoma\b|\bgbm\b|\bgrade\s*(?:iv|4)\b|\bwho\s*(?:grade\s*)?(?:iv|4)\b"
CTRL_RE = r"non[\s-]?tumou?r|\bnormal\b|\bcontrol\b|epilep"
LOW_RE = r"\bgrade\s*(?:i|ii|iii|1|2|3)\b|\bwho\s*(?:grade\s*)?(?:i|ii|iii|1|2|3)\b"
HIST_COLS = ["histology", "diagnosis", "disease_state", "disease", "tissue", "source_name_ch1", "title"]
EXPECTED = {"GSE4290": (77, 23), "GSE50161": (34, 13)}     # (GBM, control) seen in NB02; used as a check only


def classify_geo(meta):
    """Same rule as NB02: explicit GBM or control wording only; contradictions are 'ambiguous'."""
    cols = [c for c in meta.columns if c in HIST_COLS or any(k in c.lower() for k in ("characteristics", "histolog", "grade", "diagnos"))]
    require(bool(cols), f"No phenotype columns found: {list(meta.columns)}")
    txt = meta[cols].fillna("").astype(str).agg(" | ".join, axis=1).str.lower()
    gbm, ctl, low = (txt.str.contains(p, regex=True) for p in (GBM_RE, CTRL_RE, LOW_RE))
    grp = pd.Series("other", index=meta.index, dtype=object)
    grp[ctl & ~gbm] = "control"
    grp[gbm & ~ctl & ~low] = "GBM"
    grp[(gbm & ctl) | (gbm & low)] = "ambiguous"
    return grp, txt


geo_sheets = {}
for acc in CFG["geo_microarray"]:
    path = RAW / "GEO" / f"{acc}_series_matrix.txt.gz"
    require_files([path], "Run NB01 first.")
    meta, _ = read_geo_series_matrix(path)
    grp, txt = classify_geo(meta)
    sheet = pd.DataFrame({"sample_id": meta.index, "cohort": acc, "group": grp.values, "raw_label": txt.values,
                          "idh_status": "not_annotated", "primary_recurrent": "unknown", "age": np.nan, "sex": "unknown",
                          "os_time_days": np.nan, "os_event": np.nan})
    sheet["included"] = sheet.group.isin(["GBM", "control"])
    sheet["exclusion_reason"] = np.where(sheet.included, "", "non-GBM or ambiguous histology")
    n_gbm, n_ctl = int((sheet.group == "GBM").sum()), int((sheet.group == "control").sum())
    print(f"{acc}: GBM={n_gbm}, control={n_ctl}, other/ambiguous={int((~sheet.included).sum())}")
    print(sheet.groupby("group").raw_label.agg(lambda s: s.str.slice(0, 70).value_counts().head(3).to_dict()).to_string(), "\n")
    if (n_gbm, n_ctl) != EXPECTED[acc]:
        log(f"{acc}: counts {(n_gbm, n_ctl)} differ from the NB02 run {EXPECTED[acc]}; check the label table above.", "WARN")
    require(n_gbm >= MIN_N and n_ctl >= MIN_N, f"{acc}: too few GBM ({n_gbm}) or controls ({n_ctl}).")
    sheet.to_csv(P("meta", f"{acc}_samples.csv"), index=False)
    sheet[["sample_id", "raw_label", "group"]].rename(columns={"raw_label": "combined_metadata", "group": "assigned_group"}) \
        .to_csv(P("meta", f"{acc}_label_mapping_audit.csv"), index=False)
    geo_sheets[acc] = sheet
manifest_update("nb04_repair", "geo_sheets", {k: v.group.value_counts().to_dict() for k, v in geo_sheets.items()})

## A2. TCGA age and IDH status

If the automatic sources fail, download the TCGA GBM clinical and mutation tables from cBioPortal
(study *Glioblastoma Multiforme, TCGA PanCancer Atlas*) and save one CSV at
`02_Raw_Datasets/TCGA/tcga_gbm_age_idh_manual.csv` with columns `patient, age, idh_status`
(`idh_status` = `wildtype`, `mutant` or `unknown`; `patient` like `TCGA-02-0001`). Then re-run this cell.

In [ ]:
import requests

TCGA_DIR = RAW / "TCGA"; TCGA_DIR.mkdir(parents=True, exist_ok=True)
HOTSPOT = re.compile(r"R132|R172|R140")
GDC_API = "https://api.gdc.cancer.gov"
CBIO = "https://www.cbioportal.org/api"


def _post(url, body, timeout=120):
    r = requests.post(url, json=body, timeout=timeout, headers={"Content-Type": "application/json"})
    r.raise_for_status()
    return r.json()


def _get(url, params=None, timeout=120):
    r = requests.get(url, params=params, timeout=timeout, headers={"Accept": "application/json"})
    r.raise_for_status()
    return r.json()


def from_manual():
    f = TCGA_DIR / "tcga_gbm_age_idh_manual.csv"
    if not f.exists():
        return None
    d = pd.read_csv(f)
    require_columns(d, ["patient", "age", "idh_status"], f.name)
    d["patient"] = d.patient.astype(str).str.upper().str[:12]
    d["idh_status"] = d.idh_status.astype(str).str.lower().replace({"wt": "wildtype", "mut": "mutant"})
    d["source"] = "manual file"
    return d[["patient", "age", "idh_status", "source"]]


def from_gdc():
    proj = {"op": "in", "content": {"field": "project.project_id", "value": ["TCGA-GBM"]}}
    cases = _post(f"{GDC_API}/cases", {"filters": proj, "size": 3000, "format": "JSON",
                                       "fields": "submitter_id,demographic.age_at_index,diagnoses.age_at_diagnosis"})
    (TCGA_DIR / "gdc_cases_age.json").write_text(json.dumps(cases))
    rows = []
    for h in cases["data"]["hits"]:
        age = (h.get("demographic") or {}).get("age_at_index")
        if age is None:
            d0 = (h.get("diagnoses") or [{}])[0].get("age_at_diagnosis")
            age = d0 / 365.25 if d0 is not None else None
        rows.append({"patient": h["submitter_id"].upper(), "age": age})
    ages = pd.DataFrame(rows)
    files = _post(f"{GDC_API}/files", {"size": 5000, "format": "JSON", "fields": "cases.submitter_id",
                                       "filters": {"op": "and", "content": [
                                           {"op": "in", "content": {"field": "cases.project.project_id", "value": ["TCGA-GBM"]}},
                                           {"op": "in", "content": {"field": "data_type", "value": ["Masked Somatic Mutation"]}}]}})
    (TCGA_DIR / "gdc_mutation_files.json").write_text(json.dumps(files))
    sequenced = {c["submitter_id"].upper() for h in files["data"]["hits"] for c in h.get("cases", [])}
    occ = _post(f"{GDC_API}/ssm_occurrences", {"size": 2000, "format": "JSON",
                "fields": "case.submitter_id,ssm.consequence.transcript.gene.symbol,ssm.consequence.transcript.aa_change",
                "filters": {"op": "and", "content": [
                    {"op": "in", "content": {"field": "case.project.project_id", "value": ["TCGA-GBM"]}},
                    {"op": "in", "content": {"field": "ssm.consequence.transcript.gene.symbol", "value": ["IDH1", "IDH2"]}}]}})
    (TCGA_DIR / "gdc_idh_occurrences.json").write_text(json.dumps(occ))
    variants = {}
    for h in occ["data"]["hits"]:
        pid = h["case"]["submitter_id"].upper()
        for cq in (h.get("ssm") or {}).get("consequence", []):
            t = cq.get("transcript") or {}
            if (t.get("gene") or {}).get("symbol") in ("IDH1", "IDH2") and t.get("aa_change"):
                variants.setdefault(pid, set()).add(f"{t['gene']['symbol']} {t['aa_change']}")
    require(len(sequenced) > 50, f"GDC returned only {len(sequenced)} sequenced TCGA-GBM cases; treating source as failed.")
    return ages, sequenced, variants, "GDC API"


def from_cbioportal():
    st = "gbm_tcga_pan_can_atlas_2018"
    clin = _get(f"{CBIO}/studies/{st}/clinical-data", {"clinicalDataType": "PATIENT", "attributeId": "AGE", "projection": "SUMMARY"})
    ages = pd.DataFrame({"patient": [c["patientId"].upper() for c in clin], "age": [pd.to_numeric(c["value"], errors="coerce") for c in clin]})
    seq_ids = _get(f"{CBIO}/sample-lists/{st}_sequenced/sample-ids")
    sequenced = {s.upper()[:12] for s in seq_ids}
    variants = {}
    for entrez, sym in [(3417, "IDH1"), (3418, "IDH2")]:
        muts = _get(f"{CBIO}/molecular-profiles/{st}_mutations/mutations",
                    {"sampleListId": f"{st}_sequenced", "entrezGeneId": entrez, "projection": "SUMMARY"})
        for m in muts:
            variants.setdefault(m["patientId"].upper(), set()).add(f"{sym} {m.get('proteinChange', '')}")
    (TCGA_DIR / "cbioportal_age_idh.json").write_text(json.dumps({"n_age": len(ages), "n_sequenced": len(sequenced),
                                                                   "variants": {k: sorted(v) for k, v in variants.items()}}))
    require(len(sequenced) > 50, "cBioPortal returned too few sequenced samples.")
    return ages, sequenced, variants, "cBioPortal (gbm_tcga_pan_can_atlas_2018)"


def build_status(ages, sequenced, variants, source):
    pats = sorted(set(ages.patient) | sequenced | set(variants))
    out = pd.DataFrame({"patient": pats})
    out["age"] = out.patient.map(ages.drop_duplicates("patient").set_index("patient").age)
    out["idh_variants"] = out.patient.map(lambda p: ";".join(sorted(variants.get(p, []))))
    hot = out.idh_variants.str.contains(HOTSPOT)
    out["idh_status"] = np.where(hot, "mutant", np.where(out.patient.isin(sequenced), "wildtype", "unknown"))
    out["source"] = source
    return out


tcga = pd.read_csv(P("meta", "TCGA_samples.csv"))
tcga["patient"] = tcga.sample_id.astype(str).str.upper().str[:12]
status, errors = None, []
status = from_manual()
if status is None:
    for fn in (from_gdc, from_cbioportal):
        try:
            status = build_status(*fn())
            break
        except Exception as exc:          # noqa: BLE001 (try the next source; every failure is reported)
            errors.append(f"{fn.__name__}: {exc}")
            print("source failed ->", errors[-1][:300])
if status is None:
    raise ActionableError("Could not obtain TCGA age/IDH from GDC or cBioPortal.\n  " + "\n  ".join(errors) +
                          "\nUse the manual file described in the markdown above, then re-run this cell.")
status.to_csv(TCGA_DIR / "tcga_gbm_age_idh_resolved.csv", index=False)
print("Source used:", status.source.iloc[0], "| patients:", len(status), "|", status.idh_status.value_counts().to_dict())

st_map = status.drop_duplicates("patient").set_index("patient")
tcga["age"] = tcga.patient.map(st_map.age).astype(float)
tcga["idh_status"] = tcga.patient.map(st_map.idh_status).fillna("unknown")
tcga["idh_variants"] = tcga.patient.map(st_map.idh_variants) if "idh_variants" in st_map else ""
tcga["idh_source"] = status.source.iloc[0]


def tcga_reason(r):
    if r.idh_status == "mutant":
        return "IDH-mutant"
    if r.idh_status != "wildtype":
        return "IDH wildtype not verified (not sequenced)"
    if pd.isna(r.age):
        return "age unknown"
    if r.age < 18:
        return "age < 18"
    if pd.isna(r.os_time_days) or pd.isna(r.os_event):
        return "no overall-survival data"
    if r.os_time_days <= 0:
        return "non-positive survival time"
    if r.os_event not in (0, 1):
        return "invalid survival event"
    return ""


tcga["survival_exclusion_reason"] = tcga.apply(tcga_reason, axis=1)
tcga["included_survival"] = tcga.survival_exclusion_reason.eq("")
tcga["included"] = tcga.idh_status.ne("mutant")                  # expression analyses: drop known IDH-mutant only
tcga["exclusion_reason"] = np.where(tcga.included, "", "IDH-mutant")
tcga.to_csv(P("meta", "TCGA_samples.csv"), index=False)

ex_path = P("meta", "exclusion_log.csv")
ex = pd.read_csv(ex_path) if ex_path.exists() else pd.DataFrame(columns=["cohort", "sample_id", "reason"])
ex = ex[~ex.cohort.isin(["TCGA-survival", "GSE4290", "GSE50161"])]
new = [("TCGA-survival", r.sample_id, r.survival_exclusion_reason) for r in tcga.itertuples() if not r.included_survival]
for acc, sh in geo_sheets.items():
    new += [(acc, r.sample_id, r.exclusion_reason) for r in sh.itertuples() if not r.included]
ex = pd.concat([ex, pd.DataFrame(new, columns=["cohort", "sample_id", "reason"])], ignore_index=True)
ex.to_csv(ex_path, index=False)
print("TCGA primary tumours:", len(tcga), "| survival cohort (adult, IDH-wt, with OS):", int(tcga.included_survival.sum()))
print(tcga.survival_exclusion_reason.replace("", "INCLUDED").value_counts().to_string())
require(tcga.included_survival.sum() >= 50, "Fewer than 50 TCGA patients pass the survival rules; check the table above.")
manifest_update("nb04_repair", "tcga", {"source": status.source.iloc[0], "n_survival": int(tcga.included_survival.sum()),
                                        "idh": tcga.idh_status.value_counts().to_dict(), "age_missing": int(tcga.age.isna().sum())})

## B. Differential expression

In [ ]:
R_DE = r'''
suppressMessages({library(limma); library(edgeR); library(data.table)})
args <- commandArgs(TRUE)
expr_f <- args[1]; meta_f <- args[2]; mode <- args[3]; out_f <- args[4]
covar <- if (length(args) >= 5 && nzchar(args[5])) strsplit(args[5], ",")[[1]] else character(0)
x <- fread(expr_f); genes <- as.character(x[[1]]); x <- as.matrix(x[, -1, with = FALSE]); rownames(x) <- genes
md <- read.csv(meta_f, stringsAsFactors = FALSE)
md <- md[md$sample_id %in% colnames(x), , drop = FALSE]
for (cv in covar) if (cv %in% colnames(md)) md <- md[!is.na(md[[cv]]) & md[[cv]] != "", , drop = FALSE]
x <- x[, md$sample_id, drop = FALSE]
x <- x[rowSums(is.na(x)) == 0, , drop = FALSE]
grp <- factor(md$group, levels = c("control", "GBM"))
cat("samples per group:", paste(names(table(grp)), table(grp), collapse = " | "), "\n")
if (any(table(grp) < 3)) stop("fewer than 3 samples in a group")
d <- data.frame(grp = grp)
for (cv in covar) if (cv %in% colnames(md)) { v <- factor(md[[cv]]); if (nlevels(v) >= 2 && all(table(v) >= 2)) d[[cv]] <- v }
design <- model.matrix(as.formula(paste("~", paste(colnames(d), collapse = " + "))), data = d)
if (mode == "voom") {
  dge <- DGEList(counts = x)
  keep <- filterByExpr(dge, design)
  dge <- calcNormFactors(dge[keep, , keep.lib.sizes = FALSE])
  v <- voom(dge, design, plot = FALSE)
  fit <- eBayes(lmFit(v, design), robust = TRUE)
} else {
  fit <- eBayes(lmFit(x, design), trend = TRUE, robust = TRUE)
}
tt <- topTable(fit, coef = "grpGBM", number = Inf, sort.by = "none")
tt$SE <- sqrt(fit$s2.post) * fit$stdev.unscaled[, "grpGBM"]
tt$gene <- rownames(tt)
write.csv(tt[, c("gene", "logFC", "SE", "t", "P.Value", "adj.P.Val", "AveExpr")], out_f, row.names = FALSE)
cat("genes tested:", nrow(tt), "| covariates used:", paste(setdiff(colnames(d), "grp"), collapse = ","), "\n")
'''
de_script = write_r("de_limma.R", R_DE)


def run_de(name, expr, sheet, mode, covars=""):
    ep = DE_IN / f"{name}_expr.csv.gz"; sp = DE_IN / f"{name}_samples.csv"
    expr.to_csv(ep); sheet.to_csv(sp, index=False)
    out = P("de", f"{name}_DE_full.csv")
    print(name, "->", run_r(de_script, [ep, sp, mode, out, covars]).strip())
    return pd.read_csv(out)


def summarise(name, de, control_type, n_gbm, n_ctl):
    sig = de[(de["adj.P.Val"] < DE["fdr"]) & (de.logFC.abs() >= DE["min_abs_log2fc"])]
    return {"cohort": name, "control_type": control_type, "n_gbm": n_gbm, "n_control": n_ctl, "genes_tested": len(de),
            "n_sig": len(sig), "n_up": int((sig.logFC > 0).sum()), "n_down": int((sig.logFC < 0).sum()),
            "fdr": DE["fdr"], "min_abs_log2fc": DE["min_abs_log2fc"]}


def volcano(de, name):
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    sig = (de["adj.P.Val"] < DE["fdr"]) & (de.logFC.abs() >= DE["min_abs_log2fc"])
    lp = -np.log10(de["P.Value"].clip(lower=1e-300))
    ax.scatter(de.logFC[~sig], lp[~sig], s=4, c="#b8bec9", alpha=.6, label="not significant")
    ax.scatter(de.logFC[sig & (de.logFC > 0)], lp[sig & (de.logFC > 0)], s=5, c="#1f3a93", label="up in GBM")
    ax.scatter(de.logFC[sig & (de.logFC < 0)], lp[sig & (de.logFC < 0)], s=5, c="#0f9d94", label="down in GBM")
    for v in (DE["min_abs_log2fc"], -DE["min_abs_log2fc"]):
        ax.axvline(v, ls=":", c="k", lw=.6)
    ax.set_xlabel("log2 fold change (GBM vs control)"); ax.set_ylabel("-log10 p"); ax.set_title(name)
    ax.legend(frameon=False, fontsize=8, markerscale=2)
    savefig(fig, f"Volcano_{name}", folder="de"); plt.close(fig)


def heatmap(expr, de, sheet, name, n_genes=50, max_samples=150):
    top = [g for g in de.reindex(de.t.abs().sort_values(ascending=False).index).gene.head(n_genes) if g in expr.index]
    cols = sheet.sample_id[sheet.sample_id.isin(expr.columns)]
    cols = cols.sample(min(len(cols), max_samples), random_state=CFG["random_seed"])
    z = zscore_df(expr.loc[top, cols], axis=1).clip(-3, 3).fillna(0)
    grp = sheet.set_index("sample_id").loc[cols, "group"].map({"GBM": "#1f3a93", "control": "#0f9d94"})
    g = sns.clustermap(z, cmap="RdBu_r", center=0, figsize=(8, 9), col_colors=grp.values, xticklabels=False,
                       yticklabels=True, cbar_kws={"label": "row z-score"})
    g.fig.suptitle(f"{name}: top {len(top)} genes by |t| (up to {max_samples} samples shown)", y=1.0)
    savefig(g.fig, f"Heatmap_{name}", folder="de"); plt.close(g.fig)


def group_summary(expr, sheet, name):
    s = sheet.set_index("sample_id"); rows = {}
    for g in ["GBM", "control"]:
        cols = [c for c in s.index[s.group == g] if c in expr.columns]
        rows[f"mean_{g}"] = expr[cols].mean(axis=1); rows[f"sd_{g}"] = expr[cols].std(axis=1)
    pd.DataFrame(rows).to_csv(P("de", f"{name}_group_expression_summary.csv"))

In [ ]:
summaries = []
control_desc = {"GSE4290": "epilepsy-surgery non-tumour cortex (GEO histology 'non-tumor')",
                "GSE50161": "normal brain as labelled in GEO ('disease state: normal')"}
for acc in CFG["geo_microarray"]:
    ep = MAT / f"{acc}_logexpr.csv.gz"
    require_files([ep], "Run NB03 first.")
    raw = pd.read_csv(ep, index_col=0)
    if float(np.nanmax(raw.values)) > CFG["preprocessing"]["array_log2_if_max_above"]:
        log(f"{acc}: matrix max > {CFG['preprocessing']['array_log2_if_max_above']}; applying log2(x+1).", "WARN")
        raw = np.log2(raw.clip(lower=0) + 1)
    expr = to_symbols(raw, acc, "log")
    sheet = geo_sheets[acc]
    sheet = sheet[sheet.included & sheet.sample_id.isin(expr.columns)]
    n_gbm, n_ctl = int((sheet.group == "GBM").sum()), int((sheet.group == "control").sum())
    print(f"{acc}: samples matched to matrix: GBM={n_gbm}, control={n_ctl}")
    require(n_gbm >= MIN_N and n_ctl >= MIN_N, f"{acc}: matrix columns do not match the sample sheet (GBM={n_gbm}, control={n_ctl}).")
    expr = expr[sheet.sample_id]
    de = run_de(acc, expr, sheet[["sample_id", "group"]], "array")
    volcano(de, acc); heatmap(expr, de, sheet, acc); group_summary(expr, sheet, acc)
    summaries.append(summarise(acc, de, control_desc[acc], n_gbm, n_ctl))

In [ ]:
toil_c = MAT / "TOIL_counts.csv.gz"
require_files([toil_c], "TOIL_counts.csv.gz is missing; re-run NB03.")
cols = pd.read_csv(toil_c, nrows=0, index_col=0).columns
t_sheet = tcga.loc[tcga.included, ["sample_id", "group", "sex"]]
g_sheet = pd.read_csv(P("meta", "GTEx_samples.csv"))[["sample_id", "group", "sex"]]
sheet = pd.concat([t_sheet, g_sheet], ignore_index=True)
sheet["sex"] = sheet.sex.astype(str).str.strip().str.lower().replace({"nan": np.nan, "unknown": np.nan, "": np.nan})
miss_sex = sheet.sex.isna().sum()
sheet = sheet[sheet.sample_id.isin(cols) & sheet.sex.notna()]
print(f"Toil DE samples: {sheet.group.value_counts().to_dict()} (dropped {miss_sex} with unknown sex; "
      f"{int((~tcga.included).sum())} IDH-mutant TCGA tumours excluded)")
counts = to_symbols(pd.read_csv(toil_c, index_col=0), "TOIL", "counts")
counts = counts[sheet.sample_id.tolist()]
de = run_de("TOIL_TCGA_vs_GTEx", counts, sheet, "voom", covars="sex")
expr = to_symbols(pd.read_csv(MAT / "TOIL_logcpm.csv.gz", index_col=0), "TOIL_logcpm", "log")
volcano(de, "TOIL_TCGA_vs_GTEx"); heatmap(expr, de, sheet, "TOIL_TCGA_vs_GTEx"); group_summary(expr, sheet, "TOIL_TCGA_vs_GTEx")
summaries.append(summarise("TOIL_TCGA_vs_GTEx", de, "post-mortem GTEx cortex (jointly reprocessed with TCGA; study confounded with group)",
                           int((sheet.group == "GBM").sum()), int((sheet.group == "control").sum())))
del counts

In [ ]:
summ = pd.DataFrame(summaries)
summ.to_csv(P("de", "DE_summary_all_cohorts.csv"), index=False)
print(summ.to_string(index=False))
manifest_update("differential_expression", "summary", summ.to_dict("records"))
print("\nNB04 complete. Next: NB05 (meta-analysis).")